# IAPT: Benchmarking Small-Object Detection on Brain MRI: A Cerebral Microbleed Case Study

---
---

## Setup

The `Setup` phase prepares the environment for the pipeline.

- **Libraries**: Installs the necessary dependancies and imports all the libraries that will be used throughout the notebook.

- **Logger Setup**: Setup the logger and ignore unnecessary nibabel warnings.

- **Google Drive Mounting**: Mounts Google Drive if its running in Colab, if not we set the `BASE_PATH` to the local data folder within our working directory.

- **Configuration Constants**: Defines all the necessary paths, file names and constants that will be used throughout the notebook.

- **Helper Functions**: Defines some helper functions used throughout the notebook.

**Why:** This is critical to ensure that all dependencies are met and that the notebook can adapt to both local and cloud (Google Colab) environments.

### Libraries

In [ ]:
# installing dependancies
!pip install nnunetv2==2.7.0 nibabel==5.4.2 tqdm scipy ipywidgets -q

# libraries
import sys
from pathlib import Path
from tqdm.auto import tqdm
import torch
import os
import logging
import subprocess
import numpy as np
import json
import nibabel as nib
from scipy.ndimage import label
import pandas as pd

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB: from google.colab import drive

### Logger Setup

In [19]:
############
# initilising and setting up the logger
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(message)s',
    force=True
)
nib.imageglobals.logger.setLevel(logging.WARNING) 
logging.getLogger('nibabel').setLevel(logging.WARNING) # supressing warnings that are being fixed in the script
logger = logging.getLogger(__name__)

### Google Drive Mounting

In [20]:
if IS_COLAB:
    DRIVE_PATH = Path("/content/drive")
    path_str = str(DRIVE_PATH)
    drive.mount(path_str)
    logger.info(f"✅ Google Drive mounted to path '{path_str}'.")
    CLOUD_BASE_PATH = DRIVE_PATH / "MyDrive" / "iAPT_nnUNet-Matthias"
    BASE_PATH = CLOUD_BASE_PATH
else:
    BASE_PATH = Path.cwd().parent / "data"
    logger.info(f"✅ Running in local directory '{BASE_PATH}'.")

INFO - ✅ Running in local directory '/Users/matthiasmifsud/Desktop/AI/Year 2/SEM2/IAPT-Cerebral-Microbleeds/data'.


### Configuration Constants

In [30]:
############
# paths / files / constants

# paths
paths = {
    "nnUNet_raw": BASE_PATH / "nnUNet_raw",
    "nnUNet_preprocessed": BASE_PATH / "nnUNet_preprocessed",
    "nnUNet_results": BASE_PATH / "nnUNet_results"
}

DATASET_PATH = paths["nnUNet_raw"] / "Dataset001_VALDO"

IMAGES_PATH: Path = DATASET_PATH / "imagesTr"
LABELS_PATH: Path = DATASET_PATH / "labelsTr"
STATS_PATH: Path = DATASET_PATH / "verification_stats"
INFERENCE_OUTPUT_PATH = BASE_PATH / "nnUNet_inference"

ALL_PATHS = list(paths.values()) + [IMAGES_PATH, LABELS_PATH, STATS_PATH, INFERENCE_OUTPUT_PATH]

# files
METADATA_FILE: str = "dataset.json"
STATS_FILE: str = "stats.json"
INFER_EVAL_FILE: str = "evaluation_results.csv"

# constants
MODALITY_SUFFIXES: dict[str, str] = {
    "T1": "0000", 
    "T2": "0001",
    "T2s": "0002"
}

DATA_TYPE: str = ".nii.gz"
SUB_PREFIX: str = "sub-"
DEVICE: str = "cuda" if IS_COLAB else "cpu"

### Helper Functions

In [22]:
############
# directory creator helper function
def create_dirs(paths: list[Path] | Path, parents: bool = True, exist_ok: bool = True) -> None:
    if isinstance(paths, Path):
        paths = [paths]

    created = []

    for path in paths:
        if not path.exists():
            created.append(str(path))
        path.mkdir(parents=parents, exist_ok=exist_ok)

    if created:
        logger.info(f"ℹ️ Created {len(created)} directorie(s): {', '.join(created)}")
    else:
        logger.info("ℹ️ All directories already exist")

# file getter helper
def get_files(nib_path: str | Path) -> list:
    if nib_path == 'image':
        path = IMAGES_PATH
    elif nib_path == 'label':
        path = LABELS_PATH
    elif isinstance(nib_path, Path) and nib_path.exists():
        path = nib_path
    else:
        raise ValueError(f"'{nib_path}' is NOT a valid value for path_type.")
    
    files = list(path.glob(f"*{DATA_TYPE}"))

    if len(files) == 0:
        logger.error("❌ No label files found.")
        return []
    return files


---

## Setting Up Environment Variables

**What it does:**
- Dynamically creates the necessary directory tree.
- Sets the `nnUNet_raw`, `nnUNet_preprocessed`, and `nnUNet_results` variables in the system environment.

**Why:** Without these variables, the nnUNet tools cannot function, as they use these paths to organize the training and inference workflows.

In [23]:
# creating and logging all directories
create_dirs(ALL_PATHS)

# setting up envars for nnUNet
for key, path in paths.items():
    os.environ[key] = str(path)

logger.info("✅ Created all necessary directories and set nnUNet environment variables.")

INFO - ℹ️ Created 3 directorie(s): /Users/matthiasmifsud/Desktop/AI/Year 2/SEM2/IAPT-Cerebral-Microbleeds/data/nnUNet_raw/Dataset001_VALDO/imagesTr, /Users/matthiasmifsud/Desktop/AI/Year 2/SEM2/IAPT-Cerebral-Microbleeds/data/nnUNet_raw/Dataset001_VALDO/labelsTr, /Users/matthiasmifsud/Desktop/AI/Year 2/SEM2/IAPT-Cerebral-Microbleeds/data/nnUNet_raw/Dataset001_VALDO/verification_stats
INFO - ✅ Created all necessary directories and set nnUNet environment variables.


---

## Affine Alignment, Dataset Conversion & Metadata Generation

**What it does:**
- **Standardization**: Converts the `original_dataset` to the nnUNet accepted format.

- **Affine Alignment**: Uses the T2* weighted image as a reference to align the label masks.

- **Metadata**: Generates `dataset.json`, defining modalities (T1, T2, T2*) and labels.

**Why:** Accurate alignment is critical for small-object detection; if the label mask is even slightly offset from the lesion in the MRI, the model will learn incorrect features. Also we need to convert the data to the nnUNet accepted format so that the commands can run flawlessly.

[!CAUTION] This section only runs if the notebook is being run locally, If its in colab then it means that the dataset has alredy been converted to nnUNet required format with the affine alignment and the generated metadata.

In [24]:
def valdo_to_nnu() -> int:
    logger.info("ℹ️ Converting Valdo Dataset and aligning labels to fit nnUNet format.")

    subjects = [f for f in ORIG_DATASET_PATH.iterdir() if f.is_dir() and f.name.startswith(SUB_PREFIX)] # 72 subjects
    
    sub_count = 0
    failed_sub = []

    for sub_dir in tqdm(subjects, desc="Saving Subjects", unit="subjects"): # iterating over all 72 subjects
        sub_id = sub_dir.name
        id = sub_id.replace(SUB_PREFIX, "")

        orig_filepaths = {
            "lbl": sub_dir / f"{sub_id}_space-T2S_CMB{DATA_TYPE}",
            "T1": sub_dir / f"{sub_id}_space-T2S_desc-masked_T1{DATA_TYPE}",
            "T2": sub_dir / f"{sub_id}_space-T2S_desc-masked_T2{DATA_TYPE}",
            "T2s": sub_dir / f"{sub_id}_space-T2S_desc-masked_T2S{DATA_TYPE}"
        }

        target_filepaths = {
            "lbl": LABELS_PATH / f"VALDO_{id}{DATA_TYPE}",
            "T1": IMAGES_PATH / f"VALDO_{id}_{MODALITY_SUFFIXES['T1']}{DATA_TYPE}",
            "T2": IMAGES_PATH / f"VALDO_{id}_{MODALITY_SUFFIXES['T2']}{DATA_TYPE}",
            "T2s": IMAGES_PATH / f"VALDO_{id}_{MODALITY_SUFFIXES['T2s']}{DATA_TYPE}"
        }

        # verifying that all 4 files exist else we have a failed subject
        if not all(file.exists() for file in orig_filepaths.values()):
            logger.warning(f"⚠️ Subject '{sub_id}' is incomplete. Missing one or more files.")
            failed_sub.append(sub_id)
            continue
        
        try:
            # saving all images to target
            for img_type in ["T1", "T2", "T2s"]:
                nib.save(nib.load(orig_filepaths[img_type]), target_filepaths[img_type])
                
            # aligning with the T2s only (since CMB are mainly detected with T2s weighting also segmentation was done on T2s)
            t2s_img = nib.load(orig_filepaths['T2s'])

            orig_lbl_img = nib.load(orig_filepaths['lbl'])

            aligned_lbl_img = nib.Nifti1Image(
                orig_lbl_img.get_fdata(),
                t2s_img.affine,
                t2s_img.header
            )
            nib.save(aligned_lbl_img, target_filepaths['lbl']) 

            sub_count += 1
            logger.debug(f"Successfully converted and aligned '{sub_id}'.")

        except Exception as e:
            logger.error(f"❌ Error processing subject '{sub_id}': {e}")
            failed_sub.append(sub_id)

    if failed_sub:
        logger.error(f"❌ Failed to process the following subjects: {failed_sub}")
        
    return sub_count

def add_metadata(dataset_size) -> None:
    metadata = { 
        "channel_names": { 
            "0": "T1", 
            "1": "T2", 
            "2": "T2star" 
        }, 
        "labels": { 
            "background": 0, 
            "microbleed": 1 
        }, 
        "numTraining": dataset_size, 
        "file_ending": DATA_TYPE 
    }

    meta_path = DATASET_PATH / METADATA_FILE
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=4)
    logger.info(f"ℹ️ Added metadata to {str(meta_path)}.")

# running
if not IS_COLAB:
    ORIG_DATASET_PATH = BASE_PATH / "original_dataset" / "Task2"

    logger.info(f"ℹ️ Starting the nnUNet dataset setup...")
    # converting source data to target data
    dataset_size = valdo_to_nnu()
    # adding metadata
    add_metadata(dataset_size)
    logger.info("✅ Successfully finished setting up the nnUNet dataset.")
        

INFO - ℹ️ Starting the nnUNet dataset setup...
INFO - ℹ️ Converting Valdo Dataset and aligning labels to fit nnUNet format.
Saving Subjects: 100%|██████████| 72/72 [01:24<00:00,  1.17s/subjects]
INFO - ℹ️ Added metadata to /Users/matthiasmifsud/Desktop/AI/Year 2/SEM2/IAPT-Cerebral-Microbleeds/data/nnUNet_raw/Dataset001_VALDO/dataset.json.
INFO - ✅ Successfully finished setting up the nnUNet dataset.


---

## GPU Verification

**What it does:** Checks for NVIDIA GPU availability and tests CUDA functionality by running a small test tensor operation.

**Why:** Training on a CPU is not possible for this scale of data. Verifying the GPU at the start prevents the notebook from failing hours into a training process with a CPU.

In [28]:
# getting nvidia smi output
try:
    nvd_out = subprocess.check_output(["nvidia-smi"]).decode()
    logger.info(f"️️ℹ️ NVIDIA-SMI Output:\n{nvd_out}")
except Exception as e:
    logger.error(f"❌ nvidia-smi not available: {e}")
    if IS_COLAB:
        raise RuntimeError("GPU not available at runtime. (Fix: Runtime -> Change runtime type -> GPU)")

# check cuda
if torch.cuda.is_available():
    logger.info(f"️️✅ GPU Detected: {torch.cuda.get_device_name(0)}")

    # testing that the GPU is functional
    x = torch.rand(1000, 1000).cuda()
    logger.info(f"️️️️✅ GPU Functioning, tensor running on: {x.device}")
else:
    logger.error("❌ No CUDA GPUs detected.")
    if IS_COLAB:
        raise RuntimeError("GPU not available at runtime. (Fix: Runtime -> Change runtime type -> GPU)")
    else:
        logger.warning("⚠️ GPU not available at runtime, only CPU available.")


ERROR - ❌ nvidia-smi not available: [Errno 2] No such file or directory: 'nvidia-smi'
ERROR - ❌ No CUDA GPUs detected.
WARNING - ⚠️ GPU not available at runtime, only CPU available.


---

## Label Verification

**What it does:**
- **Integrity Check**: Ensures label values are binary (1/0).
- **Spatial Check**: Confirms that image dimensions and affine matrices match their corresponding labels.
- **Connected Components**: Calculates the count and volume of individual microbleeds for each subject.

**Why:** These verification steps ensure that the data is correct before we start the training and inference process. This prevents us passing misleading data to the model without knowing it and then having poor results without reason.

In [26]:
class LabelVerification:
    def __init__(self):
        self.corrupt_lbls = []
        self.misaligned_lbls = set()
        self.missing_mod = set()
        self.component_stats = {}

        try:
            with open(DATASET_PATH / METADATA_FILE, 'r') as file:
                data = json.load(file)
                self.dataset_size = data.get('numTraining', 0)
        except Exception as e:
            logger.error(f"❌ Failed to load {METADATA_FILE}: {e}")
            self.dataset_size = 0

    def _lbl_val_integrity(self, sub_id:str, lbl_data) -> bool:
        unique_vals = np.unique(lbl_data)
        if lbl_data.max() > 1 or lbl_data.min() < 0: # checking label mask range is between 0 and 1
            logger.error(f"❌ Label for '{sub_id}' is corrupt.")
            self.corrupt_lbls.append(sub_id)
            return False
        return True

    def _spatial_alignment(self, sub_id:str, lbl_img) -> None:
        for mod in MODALITY_SUFFIXES.values():
            img_path = IMAGES_PATH / f"{sub_id}_{mod}{DATA_TYPE}"

            if not img_path.exists():
                logger.error(f"❌ Missing Modality: '{sub_id}' is missing '{mod}'")
                self.missing_mod.add(f"{sub_id}_{mod}")
                continue

            img = nib.load(img_path)

            # dimension check
            if img.header.get_data_shape() != lbl_img.header.get_data_shape():
                logger.error(f"❌ Dimension Mismatch: '{sub_id}' ({mod})")
                self.misaligned_lbls.add(f"{sub_id}_{mod}")

            # spatial mapping check
            if not np.allclose(img.affine, lbl_img.affine, atol=1e-3):
                logger.error(f"❌ Affine Matrix Mismatch: '{sub_id}' ({mod})")
                self.misaligned_lbls.add(f"{sub_id}_{mod}")

    def _compute_connected_component(self, sub_id: str, lbl_img, lbl_data) -> None:
        # lbl_data == 1 is a microbleed
        label_mask, bleeds_count = label(lbl_data == 1)

        bleed_volumes = []
        voxel_spacing = ()

        # positive cases
        if bleeds_count > 0:
            # calc dimentions of bleed
            voxel_spacing = lbl_img.header.get_zooms()[:3] # ignoring 4th dim (time dim for fmri
            # get vol of each voxel
            voxel_vol = np.prod(voxel_spacing)

            # get size of each bleed (in voxel)
            bleed_sizes = np.bincount(label_mask.ravel())[1:] #skipping background dim (0s)
            # converting the bleed sizes to volumes
            bleed_volumes = (bleed_sizes * voxel_vol).tolist()

        self.component_stats[sub_id] = {
            'microbleed_count': int(bleeds_count),
            'microbleed_vol_mm3': [float(i) for i in bleed_volumes],
            'total_microbleed_vol_mm3': float(np.sum(bleed_volumes)),
            'voxel_spacing': tuple(float(i) for i in voxel_spacing),
            'max_bleed_vol_mm3': float(max(bleed_volumes)) if bleed_volumes else 0.0,
            'min_bleed_vol_mm3': float(min(bleed_volumes)) if bleed_volumes else 0.0,
        }

    def verify_labels(self) -> None:
        lbl_files = get_files('label')
        lbl_size = len(lbl_files)

        # fail if we cannot extract all of the labels
        if lbl_size != self.dataset_size:
            logger.error(f"❌ Extraced only {lbl_size}/{self.dataset_size} labels.")
            return

        logger.info(f"ℹ️ Starting label verification for {self.dataset_size} subjects...")
        for lbl_path in tqdm(lbl_files, desc="Verifying Subjects", unit="subjects"):
            sub_id = lbl_path.name.replace(DATA_TYPE, "")

            try:
                lbl_img = nib.load(lbl_path)
                lbl_data = np.asanyarray(lbl_img.dataobj)
            except Exception as e:
                logger.error(f"❌ Failed to load {sub_id}: {e}")
                self.corrupt_lbls.append(sub_id)
                continue

            # label val integrity
            if not self._lbl_val_integrity(sub_id, lbl_data):
                continue

            # spatial alignment
            self._spatial_alignment(sub_id, lbl_img)

            # connected component statistics
            self._compute_connected_component(sub_id, lbl_img, lbl_data)

        if not self.corrupt_lbls and not self.misaligned_lbls and not self.missing_mod:
            logger.info("✅ All checks passed succesfully")
            return
        logger.warning(f"⚠️ Issues found: {len(self.corrupt_lbls)} corrupt labels, {len(self.misaligned_lbls)} misaligned labels, {len(self.missing_mod)} missing modalities")

    def generate_summary(self) -> None:
        if not self.component_stats:
            logger.warning("⚠️ No statistics detected.")
            return

        stats = self.component_stats
        summary = {
            'total_subjects': len(stats),
            'statistics': stats
        }

        path = STATS_PATH / STATS_FILE
        create_dirs(STATS_PATH)
        with open(path, 'w') as f:
            json.dump(summary, f, indent=4)
        logger.info(f"ℹ️ Statistics saved to {path}")


verifier = LabelVerification()

stat_filepath = STATS_PATH / STATS_FILE
if not (stat_filepath).exists():
    logger.info("ℹ️ No existing stats found. Starting verification...")
    verifier.verify_labels()
    verifier.generate_summary()
else:
    try:
        with open(stat_filepath, 'r') as f:
            data = json.load(f)
            # checking if the stats file is valid
            if data.get('statistics'):
                logger.info(f"✅ Valid stats found for {data.get('total_subjects', 0)} subjects. Skipping.")
            else:
                raise ValueError("Stats file is empty of data.")
    # rerunning verification if the stats file is not valid
    except (json.JSONDecodeError, ValueError):
        logger.warning("⚠️ Stats file was corrupted or empty. Re-running verification...")
        verifier.verify_labels()
        verifier.generate_summary()

INFO - ℹ️ No existing stats found. Starting verification...
INFO - ℹ️ Starting label verification for 72 subjects...
Verifying Subjects: 100%|██████████| 72/72 [01:28<00:00,  1.24s/subjects]
INFO - ✅ All checks passed succesfully
INFO - ℹ️ All directories already exist
INFO - ℹ️ Statistics saved to /Users/matthiasmifsud/Desktop/AI/Year 2/SEM2/IAPT-Cerebral-Microbleeds/data/nnUNet_raw/Dataset001_VALDO/verification_stats/stats.json


---

## Plan & Preprocessing

Executes the nnUNet's `nnUNetv2_plan_and_preprocess` command.

**What it does:**
- Extract a dataset fingerprint.
- Create one or more nnUNet configurations.
- Preprocess the data for those configurations.

The output is written into `nnUNet_preprocessed/DatasetXXX_Name`.

**Why:** This command analyses the dataset and generates a preprocessing plan. It computes target spacing, intensity normalisation statistics, patch sizes, and architecture configurations. It then preprocesses all 72 training cases. This perpares the data for the training stage.

[!CAUTION] This step is CPU-bound, not GPU-bound.

In [27]:
! nnUNetv2_plan_and_preprocess -d 001 --verify_dataset_integrity

Fingerprint extraction...
Dataset001_VALDO
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
Extracting dataset fingerprint: 100%|███████████| 72/72 [00:31<00:00,  2.31it/s]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [0.82400001 0.50292944 0.50292944]. 
Current patch size: (np.int64(96), np.int64(192), np.int64(128)). 
Current median shape: [186.40776699 379.61165049 309.70873786]
Attempting to find 3d_lowre

---

## Training

Starts training using the `nnUNetv2_train` for the `3d_fullres` configuration across all 5 folds.

**What it does:**
- Calls the `nnUNetv2_train` command for the `3d_fullres` configuration across all 5 folds.
- Has checkpointing implemented across each fold hence if the training gets interrupted you just rerun the cell of the fold that was interrupted and then training completes where it left off.

**Why:** The training section builds the weights for our detection model so that we can then run inference to evaluate how it performs with CMB detection. The implementation of checkpointing helps save alot of time since training is a long process and an interruption across a fold could waste hours. With checkpointing we could just re-run the cell and we would not lose any data or time as it completes where it left off.

[!CAUTION] This section requires a GPU.

### Fold 0

In [ ]:
logger.info("ℹ️ Training Fold 0...")
! nnUNetv2_train 001 3d_fullres 0 -c -device {DEVICE}

INFO - ℹ️ Starting training...
INFO - ℹ️ Training Fold 0



############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-04-06 12:20:19.500678: Using torch.compile...
2026-04-06 12:20:22.225785: do_dummy_2d_data_aug: True
2026-04-06 12:20:22.231211: Creating new 5-fold cross-validation split...
2026-04-06 12:20:22.244171: Desired fold for training: 0
2026-04-06 12:20:22.247833: This split has 4 training and 1

INFO - ℹ️ Training Fold 1


Traceback (most recent call last):
  File "/usr/local/bin/nnUNetv2_train", line 5, in <module>
    from nnunetv2.run.run_training import run_training_entry
  File "/usr/local/lib/python3.12/dist-packages/nnunetv2/run/run_training.py", line 7, in <module>
    import torch.cuda
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 430, in <module>
    _load_global_deps()
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 381, in _load_global_deps
    _preload_cuda_deps()
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 348, in _preload_cuda_deps
    _preload_cuda_lib(lib_folder, lib_name)
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 318, in _preload_cuda_lib
    ctypes.CDLL(lib_path)
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


INFO - ℹ️ Training Fold 2


^C
Traceback (most recent call last):
  File "/usr/local/bin/nnUNetv2_train", line 5, in <module>
    from nnunetv2.run.run_training import run_training_entry
  File "/usr/local/lib/python3.12/dist-packages/nnunetv2/run/run_training.py", line 12, in <module>
    from nnunetv2.run.load_pretrained_weights import load_pretrained_weights
  File "/usr/local/lib/python3.12/dist-packages/nnunetv2/run/load_pretrained_weights.py", line 2, in <module>
  File "/usr/local/lib/python3.12/dist-packages/torch/_dynamo/__init__.py", line 76, in <module>
    from .polyfills import loader as _  # usort: skip # noqa: F401
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/_dynamo/polyfills/loader.py", line 32, in <module>
    POLYFILLED_MODULES: tuple["ModuleType", ...] = tuple(
                                                   ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/_dynamo/polyfills/loader.py", line 33, in <genexpr>
    importlib.import_mod

INFO - ℹ️ Training Fold 3
INFO - ℹ️ Training Fold 4


^C
^C


### Fold 1

In [ ]:
logger.info("ℹ️ Training Fold 1...")
! nnUNetv2_train 001 3d_fullres 1 -c -device {DEVICE}

### Fold 2

In [ ]:
logger.info("ℹ️ Training Fold 2...")
! nnUNetv2_train 001 3d_fullres 2 -c -device {DEVICE}

### Fold 3

In [ ]:
logger.info("ℹ️ Training Fold 3...")
! nnUNetv2_train 001 3d_fullres 3 -c -device {DEVICE}

### Fold 4

In [ ]:
logger.info("ℹ️ Training Fold 4...")
! nnUNetv2_train 001 3d_fullres 4 -c -device {DEVICE}

---

## Inference

**What it does:**
- Uses the `nnUNetv2_predict` command to generate segmentation masks for the test subjects.
- Applies the ensemble of weights learned during the 5-fold cross-validation to produce the most stable prediction.
- Saves the resulting predictions as NIfTI files in the `nnUNet_inference` directory.

**Why:** With inference we can now test the trained model and from this we can evaluate its performance to see if the trained model is actually good at detecting CMB.

[!CAUTION] This section requires a GPU.

In [ ]:
logger.info("ℹ️ Starting inference...")

! nnUNetv2_predict -i {str(IMAGES_PATH)} \
                   -o {str(INFERENCE_OUTPUT_PATH)} \
                   -d 001 \
                   -c 3d_fullres \
                   -f 0 1 2 3 4 --save_probabilities

---

## Evaluation

**What it does:**
- **Metric Calculation**: Computes the Lesion-Level Sensitivity (Recall), False Positives, Lesion-Level F1-Score and Voxel Dice Coefficient.

- **Exporting**: Exports results into a CSV file `evaluation_results.csv` to identify the model performance across each subject.

**Why:** This determines the performance of the trained model for each subject and we could go to conclusions on whether or not the model is working as expected.

In [31]:
def evaluate_subject(actual_lbl_path: Path):
    true_filename = actual_lbl_path.name
    pred_lbl_path = INFERENCE_OUTPUT_PATH / true_filename
    sub_id = true_filename.replace(DATA_TYPE, "")

    if not pred_lbl_path.exists():
        logger.warning(f"⚠️ No prediction label found for subject '{sub_id}'.")
        return

    actual_lbl_img = nib.load(actual_lbl_path)
    actual_data = actual_lbl_img.get_fdata() > 0 # array test data label mask
    
    pred_lbl_img = nib.load(pred_lbl_path)
    pred_data = pred_lbl_img.get_fdata() > 0 # array prediction label mask

    # connected components extraction
    actual_labeled, actual_num = label(actual_data)
    pred_labeled, pred_num = label(pred_data)

    # lesion level sensitivity (recall)
    overlap_true_lbl = np.unique(actual_labeled[pred_data])
    overlap_true_lbl = overlap_true_lbl[overlap_true_lbl > 0] # including only the microbleeds (no background)

    detected_true_count = len(overlap_true_lbl) # good predictions
    ll_sens = detected_true_count / actual_num if actual_num > 0 else 1.0 # recall

    # false positive calc
    overlap_pred_lbl = np.unique(pred_labeled[actual_data])
    overlap_pred_lbl = overlap_pred_lbl[overlap_pred_lbl > 0] # including only the microbleeds (no background)

    detected_pred_count = len(overlap_pred_lbl) # good predictions
    fp = pred_num - detected_pred_count

    # lesion level f1-score
    ll_prec = detected_pred_count / pred_num if pred_num > 0 else 1.0

    ll_f1 = (2 * ll_prec * ll_sens) / (ll_prec + ll_sens) if (ll_prec + ll_sens) > 0 else 0.0
    if actual_num == 0 and pred_num == 0: ll_f1 = 1.0 # correctly predicting a negative case

    # voxel calc
    intersection = np.logical_and(actual_data, pred_data).sum()
    vox_dice = (2.0 * intersection) / (actual_data.sum() + pred_data.sum() + 1e-6)

    return {
        "sub_id": sub_id,
        "lesion_level_sens": ll_sens,
        "false_positives": fp,
        "lesion_level_f1": ll_f1,
        "voxel_dice_coeff": vox_dice
    }

def calc_and_save_eval():
    true_labels = get_files('label')
    
    eval_results = []
    for true_lbl in true_labels:
        eval_results.append(evaluate_subject(true_lbl))
    
    df = pd.DataFrame(eval_results)
    path = INFERENCE_OUTPUT_PATH / INFER_EVAL_FILE
    df.to_csv(path, index=False)
    logger.info(f"✅ Inference evaluation complete. Saved to {path}")
    return df
    
df = calc_and_save_eval()
display(df.head())

WARNING - ⚠️ No prediction label found for subject 'VALDO_224'.
WARNING - ⚠️ No prediction label found for subject 'VALDO_324'.
WARNING - ⚠️ No prediction label found for subject 'VALDO_228'.
WARNING - ⚠️ No prediction label found for subject 'VALDO_212'.
WARNING - ⚠️ No prediction label found for subject 'VALDO_312'.
WARNING - ⚠️ No prediction label found for subject 'VALDO_111'.
WARNING - ⚠️ No prediction label found for subject 'VALDO_103'.
WARNING - ⚠️ No prediction label found for subject 'VALDO_101'.
WARNING - ⚠️ No prediction label found for subject 'VALDO_234'.
WARNING - ⚠️ No prediction label found for subject 'VALDO_226'.
WARNING - ⚠️ No prediction label found for subject 'VALDO_326'.
WARNING - ⚠️ No prediction label found for subject 'VALDO_210'.
WARNING - ⚠️ No prediction label found for subject 'VALDO_310'.
WARNING - ⚠️ No prediction label found for subject 'VALDO_302'.
WARNING - ⚠️ No prediction label found for subject 'VALDO_202'.
WARNING - ⚠️ No prediction label found f

,0
0,None
1,None
2,None
3,None
4,None


##
---
---